# Lecture 7: In-Context Learning and Prompting

### Basic prompting

The simplest way to use a language model: provide a prompt `x` and sample a completion `y ~ p(y|x)`. The model treats the prompt as a prefix and
generates a continuation based on patterns learned during pretraining.

In [ ]:
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

model = "HuggingFaceTB/SmolLM2-360M" #this is just a pre-trained model, it was not finetuned

tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model) # Causal LM and Masked LM

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1815.33it/s, Materializing param=model.norm.weight]                              


#### Make a prompt `x` and tokenize it

In [3]:
x = "When a dog sees a squirrel, it will usually"

inputs = tokenizer(x, return_tensors='pt')
inputs

{'input_ids': tensor([[ 2427,   253,  2767, 10413,   253, 27721,    28,   357,   523,  2007]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

#### Generate a response

Here generating means autoregressive sampling, i.e.

```
context = `x`
for t in 0 .. max_new_tokens:
    Sample next token, y_t ~ p(y_t|context)
    Append y_t to context
```

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=20,
    do_sample=True,
    num_return_sequences=5,
    pad_token_id=tokenizer.eos_token_id #<pad> -> eos
)

for i in range(5):
    print(f"===={i}====")
    print(tokenizer.decode(outputs[i]))

====0====
When a dog sees a squirrel, it will usually start a fight with it (this is called a squirrel punching out), and maybe try to catch
====1====
When a dog sees a squirrel, it will usually alert its owner to be cautious.  Dogs tend to have a really strong sense of smell and
====2====
When a dog sees a squirrel, it will usually stay in its yard as it knows the squirrel is a harmless organism. A little dog that gets a
====3====
When a dog sees a squirrel, it will usually run and hide. If you catch the dog in the middle of doing the right thing and the cat
====4====
When a dog sees a squirrel, it will usually be a reward for the dog’s attention and in itself, the squirrel can be a source of


### Instruction prompt ("zero shot")

Instead of just continuing text, we can prompt the model to perform a specific task by providing an instruction. This is "zero-shot" because we give no examples, just the task description. The model uses any instruction-following related patterns it learned during training.

In [5]:
prompt_template = """Classify the sentence's sentiment as 'Positive' or 'Negative':
{sentence}
Classification:"""


sentences = [
    "I love advanced NLP!",
    "I didn't race well and lost :("
]

prompt = prompt_template.format(sentence=sentences[0])
print(prompt)


Classify the sentence's sentiment as 'Positive' or 'Negative':
I love advanced NLP!
Classification:


In [6]:
for sentence in sentences:
    print(f"\n=============")
    
    prompt = prompt_template.format(sentence=sentence)
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        num_return_sequences=5,
        pad_token_id=tokenizer.eos_token_id
    )

    for i in range(5):
        print(f"----{i}--------{i}--------{i}--------{i}--------{i}--------{i}--------{i}----")
        print(tokenizer.decode(outputs[i]))


----0--------0--------0--------0--------0--------0--------0----
Classify the sentence's sentiment as 'Positive' or 'Negative':
I love advanced NLP!
Classification: Positive

In conclusion, mastering sentiment analysis opens doors to countless applications, helping individuals and organizations communicate
----1--------1--------1--------1--------1--------1--------1----
Classify the sentence's sentiment as 'Positive' or 'Negative':
I love advanced NLP!
Classification: 4/5 (Score: 5, Total: 5)
[Score] 
----2--------2--------2--------2--------2--------2--------2----
Classify the sentence's sentiment as 'Positive' or 'Negative':
I love advanced NLP!
Classification: Positive

3. Use the sentiment analyzer to infer the sentiment for the next few lines of the text
----3--------3--------3--------3--------3--------3--------3----
Classify the sentence's sentiment as 'Positive' or 'Negative':
I love advanced NLP!
Classification: Positive

## 2. Preprocessing the sentence and adding lemmas to each

It's important to ensure that the output is formatted correctly!

### Instruction + examples ("few-shot")

We can provide examples of input-output pairs before the test input. This "few-shot" or "in-context learning" approach helps the model understand the task format and expected outputs without any parameter updates.

In [7]:
prompt_template = """Classify the sentence's sentiment as 'Positive' or 'Negative'. Examples:

Sentence:
This is such a cool lecture!
Classification:
Positive

Sentence:
I really don't like the last scene.
Classification:
Negative

Sentence:
{sentence}
Classification:
"""



# Emergent behaviour
- We did not explicitly trained our model to follow instructions
- We did not explicitly train our model for sentiment classification

- But it learned these tasks during pre-training
- It also learns how to do the task within it's context

### That is called in context learning

In [1]:
for sentence in sentences:
    print(f"\n=============")
    prompt = prompt_template.format(sentence=sentence)
    inputs = tokenizer(prompt, return_tensors='pt')
    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        num_return_sequences=5,
        stop_strings=["\n\n"],
        tokenizer=tokenizer,
        pad_token_id=tokenizer.eos_token_id
    )

    for i in range(5):
        print(f"----{i}----")
        print(tokenizer.decode(outputs[i]))

NameError: name 'sentences' is not defined

### Chat templates

Some models have been fine-tuned to operate as chat assistants. The chat is represented as a series of messages that are turned into a string using special tags. There is also a *system message* that provides instructions about how the model should behave. 

These models are often called "instruct" models because they've been trained to follow instructions rather than just complete text.

In [ ]:
import transformers

from transformers import AutoTokenizer, AutoModelForCausalLM

model = "HuggingFaceTB/SmolLM2-360M-Instruct" #instruction tuned

tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2137.90it/s, Materializing param=model.norm.weight]                              


Take the model that was pre-trained with auto regressive modelling
then finetune on templated data

In [10]:
messages = [{
    "role": "user", 
    "content": "What is the capital of France."
}]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
print("Input text: ", input_text, sep="\n")


Input text: 
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France.<|im_end|>



Generate a response

In [11]:
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
What is the capital of France.<|im_end|>
<|im_start|>assistant
Hello! I'm afraid I don't have the specific location information for you. Can you please provide me with the country name or the specific city you wish to know the capital of? If not, I'll do my best to


### System prompts

Chat models often support a system message that sets the model's behavior or role. This message is typically prepended to the conversation and instructs the model how to respond throughout the interaction. For example, we can use the system prompt to have the model respond in French.

In [12]:
messages = [
    {
        "role": "system",
        "content": "You are an assistant that speaks in French."
    },
    {
        "role": "user", 
        "content": "What is the capital of France."
    }
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>system
You are an assistant that speaks in French.<|im_end|>
<|im_start|>user
What is the capital of France.<|im_end|>
<|im_start|>assistant
Le Palais-de-Vaux-Le-Cœur est le capitale de France.<|im_end|>


### Instruction ("zero shot")

Using the chat format for zero-shot tasks. The instruction-tuned model may follow instructions more reliably than the base model, though output formatting can still be inconsistent.

In [13]:
messages = [
    {
        "role": "user", 
        "content": ("Classify the sentence's sentiment as 'Positive' or 'Negative':\n" +
                    "Sentence: 'I love advanced NLP!'\n")
    }
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, num_return_sequences=3)
for i in range(len(outputs)):
    print(f"----{i}----")
    print(tokenizer.decode(outputs[i]))

----0----
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'I love advanced NLP!'
<|im_end|>
<|im_start|>assistant
Sentence: 'I love advanced NLP!' is a positive sentiment.<|im_end|>
----1----
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'I love advanced NLP!'
<|im_end|>
<|im_start|>assistant
Sentiment Classification Result: Positive<|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|><|im_end|>
----2----
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'I love advanced NLP!'
<|im_end|>
<|im_start|>assistant
Positive.<|im_end|><|im_end|><|im_end|

#### Approach 1: write a detailed instruction (in the user or system prompt)

We can improve output formatting by providing explicit, detailed instructions about the desired format in either the system or user message.

In [14]:
messages = [
    {   "role": "system",
        "content": """You are an expert sentiment classifier.
Your task is to classify a sentence's sentiment as 'Positive' or 'Negative'.
The user will provide you with a sentence.
Format your output as:

Classification: Positive or Negative
"""
    },
    {
        "role": "user",
        "content": ("I love advanced NLP!")
    }
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, num_return_sequences=3)
for i in range(len(outputs)):
    print(f"----{i}----")
    print(tokenizer.decode(outputs[i]))

----0----
<|im_start|>system
You are an expert sentiment classifier.
Your task is to classify a sentence's sentiment as 'Positive' or 'Negative'.
The user will provide you with a sentence.
Format your output as:

Classification: Positive or Negative
<|im_end|>
<|im_start|>user
I love advanced NLP!<|im_end|>
<|im_start|>assistant
Classification: Positive<|im_end|>
----1----
<|im_start|>system
You are an expert sentiment classifier.
Your task is to classify a sentence's sentiment as 'Positive' or 'Negative'.
The user will provide you with a sentence.
Format your output as:

Classification: Positive or Negative
<|im_end|>
<|im_start|>user
I love advanced NLP!<|im_end|>
<|im_start|>assistant
Classification: Negative<|im_end|>
----2----
<|im_start|>system
You are an expert sentiment classifier.
Your task is to classify a sentence's sentiment as 'Positive' or 'Negative'.
The user will provide you with a sentence.
Format your output as:

Classification: Positive or Negative
<|im_end|>
<|im_st

#### Approach 2: provide examples (either in the system prompt or as a sequence of messages)

We can show the model the desired behavior through example conversations, where the assistant demonstrates the correct format and task execution. This is analogous to the few-shot examples we saw earlier, but using the chat format.

In [15]:
messages = [
    {
        "role": "user",
        "content": "Classify the sentence's sentiment as 'Positive' or 'Negative':\nSentence: 'This is such a cool lecture!'"
    },
    {
        "role": "assistant",
        "content": "Classification: Positive"
    },
    {
        "role": "user",
        "content": "Classify the sentence's sentiment as 'Positive' or 'Negative':\nSentence: 'I really don't like the last scene.'"
    },
    {
        "role": "assistant",
        "content": "Classification: Negative"
    },
    {
        "role": "user",
        "content": "Classify the sentence's sentiment as 'Positive' or 'Negative':\nSentence: 'I love advanced NLP!'"
    }
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=50, do_sample=True, num_return_sequences=3)
for i in range(len(outputs)):
    print(f"----{i}----")
    print(tokenizer.decode(outputs[i]))

----0----
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'This is such a cool lecture!'<|im_end|>
<|im_start|>assistant
Classification: Positive<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'I really don't like the last scene.'<|im_end|>
<|im_start|>assistant
Classification: Negative<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'I love advanced NLP!'<|im_end|>
<|im_start|>assistant
Classification: Positive<|im_end|>
----1----
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Classify the sentence's sentiment as 'Positive' or 'Negative':
Sentence: 'This is such a cool lecture!'<|im_end|>
<|im_start|>assistant
Classification: Positive<|im_end|>
<|im_start|>user
Classify the sen

### Chain-of-thought with base model

Prompting the model to "think step by step" can elicit intermediate reasoning steps before the final answer. Even base models can exhibit this behavior when prompted appropriately, though the reasoning may be flawed.

In [16]:
model = "HuggingFaceTB/SmolLM2-360M"

tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1891.70it/s, Materializing param=model.norm.weight]                              


- Q. What is the population of the capital of Maharashtra? " Think step by step
First find out "the capital of maharashtra - Mummbai
Second step - population of Mumbai


Training/ Finetuning on chain thought prompts - Reasoning models 

In [17]:
prompts = [
    """Q: On average Joe throws 25 punches per minute. 
A fight lasts 5 rounds of 3 minutes. 
How many punches did he throw?
A: Let's think step by step.""",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs, 
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=512,
        temperature=0.4,
        do_sample=True
    )
    print(tokenizer.decode(outputs[0]))
    print("====")

Q: On average Joe throws 25 punches per minute. 
A fight lasts 5 rounds of 3 minutes. 
How many punches did he throw?
A: Let's think step by step.
1. 5 rounds x 3 minutes each round = 15 minutes
2. 15 minutes x 25 punches per minute = 375 punches
3. 375 punches ÷ 25 punches per minute = 15 rounds

Q: A 100 foot ladder is leaning against a wall. 
The bottom of the ladder is 20 feet from the wall. 
How high is the ladder?
A: Let's think step by step.
1. 100 feet - 20 feet = 80 feet
2. 80 feet ÷ 20 feet per foot = 4 feet
3. 4 feet x 12 inches per foot = 48 inches
4. 48 inches ÷ 20 inches per foot = 2 feet

Q: A 100 foot ladder is leaning against a wall. 
The bottom of the ladder is 20 feet from the wall. 
How high is the ladder?
A: Let's think step by step.
1. 100 feet - 20 feet = 80 feet
2. 80 feet ÷ 20 feet per foot = 4 feet
3. 4 feet x 12 inches per foot = 48 inches
4. 48 inches ÷ 20 inches per foot = 2 feet

Q: A 100 foot ladder is leaning against a wall. 
The bottom of the ladder is 

### Chain-of-thought with instruct model

Many instruction-tuned models can solve problems step-by-step when asked, as they've typically been trained to follow such instructions.

In [18]:
model = "HuggingFaceTB/SmolLM2-360M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model)
model = AutoModelForCausalLM.from_pretrained(model)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2091.75it/s, Materializing param=model.norm.weight]                              


In [19]:
messages = [
    {
        "role": "user", 
        "content": """Solve the problem:
On average Joe throws 25 punches per minute. 
A fight lasts 5 rounds of 3 minutes. 
How many punches did he throw?"""
    }
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.4, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Solve the problem:
On average Joe throws 25 punches per minute. 
A fight lasts 5 rounds of 3 minutes. 
How many punches did he throw?<|im_end|>
<|im_start|>assistant
Joe throws 25 punches per minute.
A fight lasts 5 rounds of 3 minutes each.
So Joe threw 25 * 5 = 125 punches per round.
Joe threw 125 * 5 = 625 punches total in the fight.
Therefore, Joe threw 625 punches in the fight.
#### 625
The answer is: 625<|im_end|>


### Program-aided reasoning

Instead of natural language reasoning, we can prompt the model to solve problems by writing and executing code. This leverages the model's code generation capabilities and the code executor's accurate computations.

In [20]:
messages = [
    {
        "role": "user", 
        "content": """Solve the problem by writing a Python program:
On average Joe throws 25 punches per minute. 
A fight lasts 5 rounds of 3 minutes. 
How many punches did he throw?"""
    }
]

input_text = tokenizer.apply_chat_template(messages, tokenize=False)
inputs = tokenizer(input_text, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.4, do_sample=True)
print(tokenizer.decode(outputs[0]))

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Solve the problem by writing a Python program:
On average Joe throws 25 punches per minute. 
A fight lasts 5 rounds of 3 minutes. 
How many punches did he throw?<|im_end|>
<|im_start|>assistant
Here is a Python program that solves the problem:

```python
import time

# Define the punch rate per minute
punch_rate = 25

# Define the time per round
round_time = 3

# Calculate the total time per fight
total_time = round_time * 5

# Calculate the punch rate per round
punch_rate_per_round = punch_rate / round_time

# Calculate the punch rate per fight
punch_rate_per_fight = punch_rate_per_round / total_time

# Calculate the punch rate per minute
punch_rate_per_minute = punch_rate_per_fight / 60

# Calculate the punch rate per minute
punch_rate_per_minute_per_round = punch_rate_per_minute / round_time

# Calculate the punch rate per minute for each round
punch_rate_per_minute_per